# Recovering a Noisy Gravitational-Wave-Like Chirp With SBI

This notebook is the main worked example for the workshop.

We will:

1. Define a prior over source parameters.
2. Simulate noisy strain data from those parameters.
3. Train neural posterior estimation with `sbi`.
4. Infer the parameters of one observed noisy signal.
5. Check the result with posterior predictive waveforms.

The waveform is deliberately lightweight and pedagogical. It is inspired by
compact binary chirps, but it is not a production LIGO/Virgo/KAGRA waveform
model.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    sys.path.insert(0, str(ROOT.parent / "src"))

import matplotlib.pyplot as plt
import torch

from sbi.inference import NPE

from gw_sbi_demo import (
    DEFAULT_PARAMETER_NAMES,
    GWConfig,
    build_prior,
    clean_chirp,
    make_observation,
    simulate,
    standardize,
    time_grid,
)
from gw_sbi_demo.plotting import plot_corner, plot_posterior_predictive, plot_signal

torch.manual_seed(7)
plt.rcParams.update({"figure.dpi": 120})


## 1. Configure the Simulator

The simulator maps source parameters to a noisy time series:

```text
theta = (chirp_mass, amplitude, merger_time, phase)
theta -> clean chirp -> detector-like coloured noise -> observed strain
```

Larger `noise_std` makes the inverse problem harder. During a workshop, this
is a good knob to turn after the first successful run.


In [ ]:
config = GWConfig(
    n_time=256,
    duration=1.0,
    noise_std=0.35,
    noise_knee_hz=35.0,
    device="cpu",
)

time = time_grid(config)
prior = build_prior(device=config.device)

true_theta = torch.tensor([32.0, 1.15, 0.74, 1.25])
x_obs = make_observation(true_theta, config=config, seed=42)
x_clean = clean_chirp(true_theta, config=config)

ax = plot_signal(time, x_obs, x_clean, title="Synthetic observed strain")
plt.show()


## 2. Draw Training Simulations

SBI learns from examples. We sample parameters from the prior, run the
simulator, and train a neural density estimator to approximate
`p(theta | x)`.

For a live session, start with `NUM_SIMULATIONS = 3000`. For a cleaner result,
use `8000` or more.


In [ ]:
NUM_SIMULATIONS = 5000

theta_train = prior.sample((NUM_SIMULATIONS,))
noise_generator = torch.Generator(device=config.device).manual_seed(2026)
x_train = simulate(theta_train, config=config, generator=noise_generator)

x_mean = x_train.mean(dim=0)
x_std = x_train.std(dim=0)
x_train_z = standardize(x_train, x_mean, x_std)
x_obs_z = standardize(x_obs, x_mean, x_std)

print(f"theta_train shape: {tuple(theta_train.shape)}")
print(f"x_train shape:     {tuple(x_train.shape)}")


Let us inspect a few random simulations. This is a useful teaching pause:
students can connect parameters to observable waveform structure before the
neural network appears.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True)
for ax, idx in zip(axes, torch.randperm(NUM_SIMULATIONS)[:4]):
    ax.plot(time, x_train[idx], color="0.25", lw=1.0)
    label = ", ".join(
        f"{name}={value:.2f}"
        for name, value in zip(DEFAULT_PARAMETER_NAMES, theta_train[idx])
    )
    ax.text(0.01, 0.88, label, transform=ax.transAxes, fontsize=8)
    ax.set_ylabel("strain")
axes[-1].set_xlabel("time [s]")
fig.suptitle("Random simulator draws")
fig.tight_layout()
plt.show()


## 3. Train Neural Posterior Estimation

`NPE` learns a conditional density estimator for `p(theta | x)`. Once trained,
it can be reused for any observation generated under the same prior and
simulator setup.


In [ ]:
inference = NPE(prior=prior)
density_estimator = inference.append_simulations(theta_train, x_train_z).train(
    training_batch_size=256,
    max_num_epochs=80,
    stop_after_epochs=10,
)
posterior = inference.build_posterior(density_estimator)


## 4. Condition On One Observation

We now ask for samples from:

```text
p(theta | x_obs)
```

This is the central SBI object. It is not one best fit; it is a distribution
over all parameter values compatible with the noisy signal and the prior.


In [ ]:
posterior_samples = posterior.sample((5000,), x=x_obs_z)

posterior_mean = posterior_samples.mean(dim=0)
posterior_std = posterior_samples.std(dim=0)

for name, truth, mean, std in zip(
    DEFAULT_PARAMETER_NAMES,
    true_theta,
    posterior_mean,
    posterior_std,
):
    print(f"{name:>12s}: truth={truth:6.3f}  posterior={mean:6.3f} +/- {std:6.3f}")


In [ ]:
fig = plot_corner(
    posterior_samples,
    labels=DEFAULT_PARAMETER_NAMES,
    truths=true_theta,
)
plt.show()


## 5. Posterior Predictive Check

A posterior is only useful if it can explain the observation. We draw parameter
samples from the posterior, generate the implied clean chirps, and overlay them
on the noisy observation.

If the observed signal lies outside this predictive cloud, the posterior may be
overconfident, the simulator may be wrong, or the observation may contain
structure that the simulator cannot express.


In [ ]:
draw_ids = torch.randperm(posterior_samples.shape[0])[:160]
predictive_clean = clean_chirp(posterior_samples[draw_ids], config=config)

ax = plot_posterior_predictive(
    time,
    x_obs,
    predictive_clean,
    title="Posterior implied clean chirps over the observed data",
)
ax.plot(time, x_clean, color="tab:orange", lw=2.0, label="true clean signal")
ax.legend(frameon=False)
plt.show()


## 6. A Simple Misspecification Stress Test

Now add a short glitch to the observation but keep using the same posterior
estimator. The point is not to get a perfect answer; the point is to show that
posterior predictive checks can expose when the simulator is missing part of
the data-generating process.


In [ ]:
glitch = 1.2 * torch.exp(-0.5 * ((time - 0.48) / 0.012) ** 2)
x_glitch = x_obs + glitch
x_glitch_z = standardize(x_glitch, x_mean, x_std)

glitch_samples = posterior.sample((3000,), x=x_glitch_z)
glitch_draw_ids = torch.randperm(glitch_samples.shape[0])[:160]
glitch_predictive = clean_chirp(glitch_samples[glitch_draw_ids], config=config)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
plot_signal(time, x_glitch, x_clean, ax=axes[0], title="Observation with an unmodelled glitch")
plot_posterior_predictive(
    time,
    x_glitch,
    glitch_predictive,
    ax=axes[1],
    title="Predictive check exposes the mismatch",
)
fig.tight_layout()
plt.show()


## Suggested Workshop Exercises

1. Change `config.noise_std` to `0.55`. Which parameters become uncertain?
2. Reduce `NUM_SIMULATIONS` to `1000`. What changes in the posterior?
3. Increase the prior width on `merger_time`. Does the posterior still find the signal?
4. Replace coloured noise with white noise in `src/gw_sbi_demo/simulator.py`.
5. Add a second chirp to the observation. Can the current simulator explain it?
